# Notebook 17: Download extra seasonal forecasts for the six stations

**Project:** SEAS5_SARIMA_CP

This notebook downloads, from the Copernicus Climate Data Store (CDS), the extra forecast information for the next experiments:

1. **ECMWF SEAS5 (system 51):** maximum and minimum 2 m temperature and rainfall (we already have mean 2 m temperature).
2. **Four other forecast systems with start dates from 1993 to 2025:** ECCC CanESM5.1p1bc (system 4), ECCC GEM5.2-NEMO (system 5), Météo-France System 9, CMCC SPS4 (system 4). For each: mean, maximum and minimum 2 m temperature and rainfall.

For each station it keeps the nearest grid cell and the average of the 3 x 3 cells around it, for every member, start month (1993 to 2025) and lead (1 to 6).

**Before you run (one time only):**
1. Make a free account at https://cds.climate.copernicus.eu and log in.
2. Open the dataset page *Seasonal forecast monthly statistics on single levels* and accept the licence at the bottom of the **Download** tab.
3. Copy your API token from your CDS profile page. In Colab, open the key icon (Secrets), add a secret named `CDS_API_KEY` with the token as its value, and switch on notebook access.

The download is resumable: if Colab disconnects, run the notebook again and it skips files already on Drive. Expect several hours, mostly waiting in the CDS queue.

In [ ]:
# 0. Setup
!pip -q install "cdsapi>=0.7.4" cfgrib eccodes
from google.colab import userdata, drive
import cdsapi, json, time, itertools
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/SEAS5_SARIMA_CP/data_raw/multisystem"); (OUT / "grib").mkdir(parents=True, exist_ok=True)
c = cdsapi.Client(url="https://cds.climate.copernicus.eu/api", key=userdata.get("CDS_API_KEY"))
STATIONS = {"KIAMO-Accra": (5.61, -0.17), "Koforidua": (6.09, -0.26), "Kumasi": (6.69, -1.62),
            "Navrongo": (10.89, -1.09), "Tamale": (9.41, -0.85), "Wa": (10.06, -2.50)}
AREA = [13, -5, 3, 3]                     # north, west, south, east: Ghana with one grid cell to spare
YEARS = [str(y) for y in range(1993, 2026)]
MONTHS = [f"{m:02d}" for m in range(1, 13)]
LEADS = ["1", "2", "3", "4", "5", "6"]
SYSTEMS = [  # (originating_centre, system, variables)
    ("ecmwf", "51", ["maximum_2m_temperature_in_the_last_24_hours", "minimum_2m_temperature_in_the_last_24_hours", "total_precipitation", "2m_temperature"]),
    ("eccc", "4", ["2m_temperature", "maximum_2m_temperature_in_the_last_24_hours", "minimum_2m_temperature_in_the_last_24_hours", "total_precipitation"]),
    ("eccc", "5", ["2m_temperature", "maximum_2m_temperature_in_the_last_24_hours", "minimum_2m_temperature_in_the_last_24_hours", "total_precipitation"]),
    ("meteo_france", "9", ["2m_temperature", "maximum_2m_temperature_in_the_last_24_hours", "minimum_2m_temperature_in_the_last_24_hours", "total_precipitation"]),
    ("cmcc", "4", ["2m_temperature", "maximum_2m_temperature_in_the_last_24_hours", "minimum_2m_temperature_in_the_last_24_hours", "total_precipitation"]),
]
print("Ready. Systems:", [f"{a} {b}" for a, b, _ in SYSTEMS])

## 1. Test request

One small request (ECMWF, January 2000) to check the account, the licence and the variable names. If it fails, the message says why; fix that before running the full download.

In [ ]:
test = OUT / "grib" / "test_ecmwf_51_2000_01.grib"
c.retrieve("seasonal-monthly-single-levels", {
    "originating_centre": "ecmwf", "system": "51", "variable": SYSTEMS[0][2], "product_type": ["monthly_mean"],
    "year": ["2000"], "month": ["01"], "leadtime_month": LEADS, "data_format": "grib", "area": AREA}, str(test))
ds = xr.open_dataset(test, engine="cfgrib", backend_kwargs={"time_dims": ("forecastMonth", "time")})
print(ds)

## 2. Full download

One request per system and start year (all 12 start months, 6 leads, all variables). Files already on Drive are skipped. If a system refuses a year (for example a start month that does not exist for that system), the failure is logged and the loop goes on.

In [ ]:
log = []
for (centre, system, vars_), year in itertools.product(SYSTEMS, YEARS):
    f = OUT / "grib" / f"{centre}_{system}_{year}.grib"
    if f.exists() and f.stat().st_size > 0: continue
    try:
        c.retrieve("seasonal-monthly-single-levels", {
            "originating_centre": centre, "system": system, "variable": vars_, "product_type": ["monthly_mean"],
            "year": [year], "month": MONTHS, "leadtime_month": LEADS, "data_format": "grib", "area": AREA}, str(f))
        log.append(dict(centre=centre, system=system, year=year, status="ok"))
    except Exception as e:
        log.append(dict(centre=centre, system=system, year=year, status="failed: " + str(e)[:200])); print(centre, system, year, "failed:", str(e)[:200])
    pd.DataFrame(log).to_csv(OUT / "download_log.csv", index=False)
print("Finished this pass. Failed requests:", sum(r["status"] != "ok" for r in log))

## 3. Extract the six stations

For each file: open every variable, take the nearest grid cell and the mean of the 3 x 3 cells around it for each station. Temperatures are converted to °C and rainfall to mm per day. Lagged-start systems (Météo-France) are read with their nominal start date.

In [ ]:
CONV = {"t2m": ("t2m_c", lambda v: v - 273.15), "mx2t24": ("tmax_c", lambda v: v - 273.15),
        "mn2t24": ("tmin_c", lambda v: v - 273.15), "tprate": ("pr_mm_day", lambda v: v * 1000 * 86400)}
def open_grib(f):
    for dims in (("forecastMonth", "indexing_time"), ("forecastMonth", "time")):
        try:
            dss = [xr.open_dataset(f, engine="cfgrib", backend_kwargs={"time_dims": dims, "filter_by_keys": {"shortName": sn}}) for sn in CONV]
            return [d for d in dss if len(d.data_vars)], dims[1]
        except Exception:
            continue
    raise RuntimeError(f"cannot read {f}")
rows = []
for f in sorted((OUT / "grib").glob("*_*_*.grib")):
    if f.name.startswith("test"): continue
    centre, system, year = f.stem.rsplit("_", 2)
    try: dss, tdim = open_grib(f)
    except Exception as e: print("skip", f.name, e); continue
    for ds in dss:
        for var in ds.data_vars:
            if var not in CONV: continue
            name, conv = CONV[var]; da = ds[var]
            for st, (lat, lon) in STATIONS.items():
                near = da.sel(latitude=lat, longitude=lon, method="nearest")
                la0, lo0 = float(near.latitude), float(near.longitude)
                box = da.sel(latitude=slice(la0 + 1.01, la0 - 1.01), longitude=slice(lo0 - 1.01, lo0 + 1.01)).mean(["latitude", "longitude"])
                for kind, arr in (("near", near), ("box3", box)):
                    df = conv(arr).to_dataframe(name="value").reset_index()
                    df["start"] = pd.to_datetime(df[tdim]); df = df.rename(columns={"number": "member", "forecastMonth": "lead"})
                    df = df.assign(centre=centre, system=system, station=st, variable=name, cell=kind)
                    rows.append(df[["centre", "system", "station", "start", "lead", "member", "variable", "cell", "value"]])
ALL = pd.concat(rows, ignore_index=True)
ALL["year"], ALL["month"] = ALL.start.dt.year, ALL.start.dt.month
ALL.drop(columns="start").to_csv(OUT / "multisystem_station_forecasts.csv.gz", index=False)
print(ALL.groupby(["centre", "system", "variable"]).agg(first=("year", "min"), last=("year", "max"), members=("member", "max"), rows=("value", "size")))

## 4. What to send back

Send the file `multisystem_station_forecasts.csv.gz` from Google Drive (`MyDrive/SEAS5_SARIMA_CP/data_raw/multisystem/`) and `download_log.csv` from the same folder. The GRIB files can stay on Drive.